# Afternoon class 30/08 — Worksheet 13 SOLUTIONS: JSON   (L03)

Every cell below was executed in the lab image (pandas 3.0.5) against the real
files in `data/`, and the quoted output is what it actually printed.

Question 4 is the one to re-read. `read_json` succeeds on the nested file and
gives you something that is not usable as a table.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 13 — JSON. Run this once.
import json
import pandas as pd

#   data/customers.json      120 flat records
#   data/orders_nested.json   80 records, each with a nested "customer" object

with open("data/orders_nested.json") as fh:
    nested_raw = json.load(fh)

print("first nested record:")
print(json.dumps(nested_raw[0], indent=2))

PART A — flat JSON

### Question 1

`(120, 5)` -> `CustomerID int64`, the rest `str`.

A list of flat objects loads exactly like a CSV, one record per row.

Note `CustomerID` arrived as `int64` without you asking. JSON distinguishes
`40732966` from `"40732966"`, so the type came from the file rather than
from Pandas guessing.

In [ ]:
cust = pd.read_json("data/customers.json")
print("shape:", cust.shape)
print("columns:", list(cust.columns))
print()
print(cust.dtypes)
print()
print(cust.head(3))

### Question 2

The file starts `[ { "CustomerID": 40732966, ... ` — a list of objects. -> Ontario `30`, West `28`, Prarie `26`, Atlantic `17`, Quebec `8`, NWT `4`, Nunavut `4`, Yukon `3`.

That layout is `orient="records"`: an array whose elements are one object
per row. It is what almost every web API returns, and `read_json` handles
it without being told.

`Prarie` is here too — the same misspelling as the CSV, because it is the
same source system. Consistent across formats, still wrong.

In [ ]:
with open("data/customers.json") as fh:
    print(fh.read(120))
print("...")
print()
cust = pd.read_json("data/customers.json")
print(cust["Region"].value_counts())

### Question 3

Both give a `DataFrame`. -> JSON carries types (`CustomerID` is `int64`); CSV is text throughout and must be inferred.

The trade is types against size. JSON records the difference between a
number and a string, so `CustomerID` cannot silently become text and a
zero-padded code cannot lose its zeros.

The cost is that JSON repeats every key on every record. `customers.json`
is 18.8 KB for 120 rows; the equivalent CSV would be a fraction of that.
On millions of rows that ratio is the reason CSV and Parquet still exist.

In [ ]:
cust = pd.read_json("data/customers.json")
sales = pd.read_csv("data/sales.csv")
print("json ->", type(cust).__name__, cust.shape)
print("csv  ->", type(sales).__name__, sales.shape)
print()
print("json dtypes:")
print(cust.dtypes)

# JSON carries types: CustomerID arrived as int64 because it is a number in
# the file. A CSV has no types at all -- everything is text and Pandas
# guesses. What CSV gives you back is size: JSON repeats every key on every
# record, so it is far larger for the same data.

PART B — nested JSON

### Question 4

`(80, 4)` -> columns `['OrderID', 'OrderDate', 'Sales', 'customer']`, and each `customer` cell is a **`dict`**.

It did not raise. You have a DataFrame with the right number of rows, and
one column whose every cell is a Python dictionary.

This is a table in shape only. The `customer` column cannot be grouped on,
plotted, filtered by region, or written to CSV in any useful way — you
cannot reach `region` without opening each dict yourself.

And because the dtype is `object`, nothing complains. `read_json` did the
only thing it could: JSON allows nesting, tables do not, so it put the
unconvertible thing in a cell verbatim and moved on.

In [ ]:
nested = pd.read_json("data/orders_nested.json")
print("shape:", nested.shape)
print("columns:", list(nested.columns))
print()
print(nested.head(3))
print()
first = nested["customer"].iloc[0]
print("type of one customer cell:", type(first).__name__)
print("its contents:", first)

### Question 5

`value_counts()` **works**, counting whole dictionaries as values. `.str.upper()` -> **all `NaN`, `dtype: float64`**, no error.

Both results are useless and neither one told you so.

`value_counts()` succeeded and is counting *entire customer objects* as if
they were categories — so 'Bill Donatelli, Ontario, Corporate' appearing 8
times is really 8 orders from one customer, and any customer whose record
differed in a single field would count as a different category.

`.str.upper()` is worse. The `.str` accessor returns `NaN` for anything
that is not a string rather than raising, so you get a column of nothing,
silently, and the dtype flips to `float64` because `NaN` is a float. A
column of `float64` `NaN` where you asked for upper-case text is a strong
signal that the values were never strings.

In [ ]:
nested = pd.read_json("data/orders_nested.json")
for label, fn in [
    ("value_counts", lambda: nested["customer"].value_counts().head(3)),
    (".str.upper()", lambda: nested["customer"].str.upper().head(3)),
]:
    try:
        print("=== %s ===" % label)
        print(fn())
    except Exception as exc:
        print("%s -> %s: %s" % (label, type(exc).__name__, exc))
    print()

### Question 6

`pd.json_normalize` -> `(80, 6)` with columns `OrderID, OrderDate, Sales, customer.name, customer.region, customer.segment`.

The nested object became three real columns, named by their path with a
dot. Two more columns than Q4 and every one of them usable.

`json_normalize` takes the parsed Python object, not a filename — which is
why the setup cell loaded the file with `json.load` first. That is the
normal shape of this work: parse with `json`, flatten with
`json_normalize`, then it is an ordinary DataFrame.

In [ ]:
flat = pd.json_normalize(nested_raw)
print("shape:", flat.shape)
print("columns:", list(flat.columns))
print()
print(flat.head(3))

### Question 7

`customer.region` -> Ontario `28`, West `24`, Atlantic `12`, Quebec `8`, NWT `5`, Prarie `3`. -> mean Sales by region ranges from `233.77` (Quebec) to `925.29` (NWT).

Now the nested field behaves like any other column: countable, groupable,
aggregatable.

Worth noting the group sizes before reading the means. Northwest
Territories has the highest average at `925.29` and it is computed from
**five** orders. Prairie's `497.36` comes from three. Those averages are
real arithmetic on real data and they are far too small a sample to say
anything about a region — one large order would move either of them
entirely.

The dotted column name needs quoting or bracket access; you cannot write
`flat.customer.region` and get the column.

In [ ]:
flat = pd.json_normalize(nested_raw)
print(flat["customer.region"].value_counts())
print()
print(flat.groupby("customer.region")["Sales"].mean().round(2).to_string())

PART C — writing JSON back out

### Question 8

`orient="records"` -> a JSON array of row objects. `orient="columns"` (the default) -> an object of objects keyed by column, then by **row label as a string**.

`records` is what other systems expect. It is the shape APIs return, the
shape most JSON parsers assume, and it survives being read by anything.

The default `columns` layout nests column name over row label and turns the
index into string keys — `"0"`, `"1"`, `"2"`. It round-trips through Pandas
perfectly and is awkward for almost every other consumer.

So the default is the one you least often want when you are handing the
file to someone else. Pass `orient="records"` unless the reader is Pandas.

In [ ]:
flat = pd.json_normalize(nested_raw)
small = flat.head(3)[["OrderID", "Sales"]]

print("=== orient='records' ===")
print(small.to_json(orient="records", indent=2))
print()
print("=== orient='columns' (the default) ===")
print(small.to_json(orient="columns", indent=2))

### Question 9

Both orientations round-trip: `.equals()` is `True` for `records` and for the default.

Both survive the trip because Pandas wrote them and Pandas read them back —
it knows how to reverse its own layouts.

That is exactly why the default feels safe and is not. The test that passes
here is Pandas-to-Pandas. The moment the consumer is a JavaScript front end
or another team's parser, `records` is the only one that reads naturally.

Also note `indent=2` is for humans only. Writing without it produces the
same data in far fewer bytes, which is what you want for anything machine
bound.

In [ ]:
flat = pd.json_normalize(nested_raw)
small = flat.head(3)[["OrderID", "Sales"]]

small.to_json("/tmp/out_records.json", orient="records")
back_r = pd.read_json("/tmp/out_records.json")
print("records round-trip equal:", small.equals(back_r))

small.to_json("/tmp/out_cols.json")
back_c = pd.read_json("/tmp/out_cols.json")
print("default round-trip equal:", small.equals(back_c))
print()
print("read back from records:")
print(back_r)

### Question 10

`pd.read_json("data/sales.csv")` -> **raises** `ValueError: Expected object or value`.

The parser looked for `{` or `[` and found `OrderID,OrderDate,...`. The
message is terse but honest.

This is the well-behaved end of the sheet. Compare it with Q4, where the
file *was* valid JSON and the failure was semantic rather than syntactic —
valid input, successful parse, unusable table. Q10 you find in one second;
Q4 you find when someone asks why the regional breakdown is empty.

In [ ]:
print(pd.read_json("data/sales.csv"))